# Day 12 — SQL Fundamentals
### SELECT · WHERE · JOINs · GROUP BY · HAVING · Subqueries

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import duckdb
import sqlite3

# Load Titanic into DuckDB (in-memory SQL engine)
df = pd.read_csv(r"C:\DS-AI-75d\titanic.csv")

# Create DuckDB connection
con = duckdb.connect()
con.register("titanic", df)

print(f"Pandas:  {pd.__version__}")
print(f"DuckDB:  {duckdb.__version__}")
print(f"Dataset: {df.shape}")
print(f"Columns: {list(df.columns)}")
print("Ready! ✅")

Pandas:  2.3.3
DuckDB:  1.5.1
Dataset: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
Ready! ✅


## 2. SELECT & WHERE

In [ ]:
def sql(query):
    """Helper function to run SQL and return DataFrame"""
    return con.execute(query).df()


print("=" * 55)
print("          SELECT & WHERE")
print("=" * 55)

# Basic SELECT
print("\n--- Q1: First 5 passengers ---")
print(
    sql(
        """
    SELECT PassengerId, Name, Age, Sex, Pclass, Survived
    FROM titanic
    LIMIT 5
"""
    )
)

# WHERE filter
print("\n--- Q2: Female passengers over 40 ---")
print(
    sql(
        """
    SELECT Name, Age, Sex, Pclass, Survived
    FROM titanic
    WHERE Sex = 'female' AND Age > 40
    ORDER BY Age DESC
    LIMIT 8
"""
    )
)

# WHERE with multiple conditions
print("\n--- Q3: 1st class survivors ---")
result = sql(
    """
    SELECT Name, Age, Fare, Survived
    FROM titanic
    WHERE Pclass = 1 AND Survived = 1
    ORDER BY Fare DESC
    LIMIT 5
"""
)
print(result)
print(
    f"\nTotal 1st class survivors: {sql('SELECT COUNT(*) FROM titanic WHERE Pclass=1 AND Survived=1').iloc[0,0]}"
)

          SELECT & WHERE

--- Q1: First 5 passengers ---
   PassengerId                                               Name   Age  \
0            1                            Braund, Mr. Owen Harris  22.0   
1            2  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0   
2            3                             Heikkinen, Miss. Laina  26.0   
3            4       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0   
4            5                           Allen, Mr. William Henry  35.0   

      Sex  Pclass  Survived  
0    male       3         0  
1  female       1         1  
2  female       3         1  
3  female       1         1  
4    male       3         0  

--- Q2: Female passengers over 40 ---
                                               Name   Age     Sex  Pclass  \
0                 Andrews, Miss. Kornelia Theodosia  63.0  female       1   
1                            Turkula, Mrs. (Hedwig)  63.0  female       3   
2         Stone, Mrs. George Nelson (Marth

## 3. GROUP BY & HAVING

In [ ]:
print("=" * 55)
print("          GROUP BY & HAVING")
print("=" * 55)

# GROUP BY — survival by class
print("\n--- Q4: Survival rate by passenger class ---")
print(
    sql(
        """
    SELECT 
        Pclass,
        COUNT(*) as total_passengers,
        SUM(Survived) as survivors,
        ROUND(AVG(Survived) * 100, 1) as survival_rate_pct,
        ROUND(AVG(Age), 1) as avg_age,
        ROUND(AVG(Fare), 2) as avg_fare
    FROM titanic
    GROUP BY Pclass
    ORDER BY Pclass
"""
    )
)

# GROUP BY with multiple columns
print("\n--- Q5: Survival by class AND gender ---")
print(
    sql(
        """
    SELECT 
        Pclass,
        Sex,
        COUNT(*) as total,
        SUM(Survived) as survived,
        ROUND(AVG(Survived) * 100, 1) as survival_pct
    FROM titanic
    GROUP BY Pclass, Sex
    ORDER BY Pclass, Sex
"""
    )
)

# HAVING — filter on aggregated results
print("\n--- Q6: Classes where avg fare > 50 (HAVING) ---")
print(
    sql(
        """
    SELECT 
        Pclass,
        COUNT(*) as passengers,
        ROUND(AVG(Fare), 2) as avg_fare,
        ROUND(AVG(Age), 1) as avg_age
    FROM titanic
    GROUP BY Pclass
    HAVING AVG(Fare) > 50
    ORDER BY avg_fare DESC
"""
    )
)

print("\n💡 WHERE filters ROWS, HAVING filters GROUPS!")

          GROUP BY & HAVING

--- Q4: Survival rate by passenger class ---
   Pclass  total_passengers  survivors  survival_rate_pct  avg_age  avg_fare
0       1               216      136.0               63.0     38.2     84.15
1       2               184       87.0               47.3     29.9     20.66
2       3               491      119.0               24.2     25.1     13.68

--- Q5: Survival by class AND gender ---
   Pclass     Sex  total  survived  survival_pct
0       1  female     94      91.0          96.8
1       1    male    122      45.0          36.9
2       2  female     76      70.0          92.1
3       2    male    108      17.0          15.7
4       3  female    144      72.0          50.0
5       3    male    347      47.0          13.5

--- Q6: Classes where avg fare > 50 (HAVING) ---
   Pclass  passengers  avg_fare  avg_age
0       1         216     84.15     38.2

💡 WHERE filters ROWS, HAVING filters GROUPS!


## 4. JOINs

In [ ]:
# Create two related tables to demonstrate JOINs
con.execute(
    """
    CREATE OR REPLACE TABLE passengers AS
    SELECT PassengerId, Name, Age, Sex, Pclass, Survived, Fare
    FROM titanic
    LIMIT 10
"""
)

con.execute(
    """
    CREATE OR REPLACE TABLE class_info AS
    SELECT * FROM (VALUES
        (1, 'First Class',  'Luxury', 'Upper Deck'),
        (2, 'Second Class', 'Standard', 'Middle Deck'),
        (3, 'Third Class',  'Basic', 'Lower Deck')
    ) t(Pclass, class_name, tier, location)
"""
)

print("=" * 55)
print("              JOINs")
print("=" * 55)

print("\n--- Passengers table (first 5) ---")
print(sql("SELECT * FROM passengers LIMIT 5"))

print("\n--- Class info table ---")
print(sql("SELECT * FROM class_info"))

# INNER JOIN
print("\n--- Q7: INNER JOIN — passengers with class info ---")
print(
    sql(
        """
    SELECT p.PassengerId, p.Name, p.Pclass,
           c.class_name, c.tier, c.location,
           p.Survived
    FROM passengers p
    INNER JOIN class_info c ON p.Pclass = c.Pclass
    ORDER BY p.PassengerId
    LIMIT 6
"""
    )
)

# LEFT JOIN
print("\n--- Q8: LEFT JOIN — keep all passengers ---")
print(
    sql(
        """
    SELECT p.PassengerId, p.Pclass,
           c.class_name,
           p.Survived
    FROM passengers p
    LEFT JOIN class_info c ON p.Pclass = c.Pclass
    LIMIT 6
"""
    )
)

print(
    """
JOIN TYPES SUMMARY:
  INNER JOIN — only rows that match in BOTH tables
  LEFT JOIN  — ALL rows from left + matches from right
  RIGHT JOIN — ALL rows from right + matches from left
  FULL JOIN  — ALL rows from both tables
"""
)

              JOINs

--- Passengers table (first 5) ---
   PassengerId                                               Name   Age  \
0            1                            Braund, Mr. Owen Harris  22.0   
1            2  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0   
2            3                             Heikkinen, Miss. Laina  26.0   
3            4       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0   
4            5                           Allen, Mr. William Henry  35.0   

      Sex  Pclass  Survived     Fare  
0    male       3         0   7.2500  
1  female       1         1  71.2833  
2  female       3         1   7.9250  
3  female       1         1  53.1000  
4    male       3         0   8.0500  

--- Class info table ---
   Pclass    class_name      tier     location
0       1   First Class    Luxury   Upper Deck
1       2  Second Class  Standard  Middle Deck
2       3   Third Class     Basic   Lower Deck

--- Q7: INNER JOIN — passengers with class i

## 5. Subqueries

In [ ]:
print("=" * 55)
print("            SUBQUERIES")
print("=" * 55)

# Subquery in WHERE
print("\n--- Q9: Passengers who paid above average fare ---")
print(
    sql(
        """
    SELECT Name, Pclass, Fare, Survived
    FROM titanic
    WHERE Fare > (SELECT AVG(Fare) FROM titanic)
    ORDER BY Fare DESC
    LIMIT 6
"""
    )
)

avg_fare = sql("SELECT ROUND(AVG(Fare),2) as avg FROM titanic").iloc[0, 0]
print(f"Average fare was: £{avg_fare}")

# Subquery in FROM
print("\n--- Q10: Class survival summary using subquery ---")
print(
    sql(
        """
    SELECT 
        class_stats.Pclass,
        class_stats.total,
        class_stats.survival_rate,
        CASE 
            WHEN class_stats.survival_rate >= 50 THEN 'Good chance'
            WHEN class_stats.survival_rate >= 30 THEN 'Moderate'
            ELSE 'Poor chance'
        END as survival_category
    FROM (
        SELECT 
            Pclass,
            COUNT(*) as total,
            ROUND(AVG(Survived)*100, 1) as survival_rate
        FROM titanic
        GROUP BY Pclass
    ) as class_stats
    ORDER BY Pclass
"""
    )
)

# Subquery with IN
print("\n--- Q11: Passengers from most expensive ticket groups ---")
print(
    sql(
        """
    SELECT Name, Fare, Pclass, Survived
    FROM titanic
    WHERE Ticket IN (
        SELECT Ticket
        FROM titanic
        GROUP BY Ticket
        HAVING AVG(Fare) > 200
    )
    ORDER BY Fare DESC
    LIMIT 6
"""
    )
)

            SUBQUERIES

--- Q9: Passengers who paid above average fare ---
                                 Name  Pclass      Fare  Survived
0                    Ward, Miss. Anna       1  512.3292         1
1  Cardeza, Mr. Thomas Drake Martinez       1  512.3292         1
2              Lesurer, Mr. Gustave J       1  512.3292         1
3                   Fortune, Mr. Mark       1  263.0000         0
4      Fortune, Mr. Charles Alexander       1  263.0000         0
5          Fortune, Miss. Mabel Helen       1  263.0000         1
Average fare was: £32.2

--- Q10: Class survival summary using subquery ---
   Pclass  total  survival_rate survival_category
0       1    216           63.0       Good chance
1       2    184           47.3          Moderate
2       3    491           24.2       Poor chance

--- Q11: Passengers from most expensive ticket groups ---
                                 Name      Fare  Pclass  Survived
0                    Ward, Miss. Anna  512.3292       1       

## 6. Practice SQL Questions

In [ ]:
print("=" * 55)
print("       PRACTICE SQL QUESTIONS")
print("=" * 55)

# Q12
print("\n--- Q12: How many passengers embarked from each port? ---")
print(
    sql(
        """
    SELECT 
        Embarked,
        COUNT(*) as passengers,
        ROUND(AVG(Survived)*100,1) as survival_pct
    FROM titanic
    WHERE Embarked IS NOT NULL
    GROUP BY Embarked
    ORDER BY passengers DESC
"""
    )
)

# Q13
print("\n--- Q13: Top 5 most common age groups ---")
print(
    sql(
        """
    SELECT 
        FLOOR(Age/10)*10 as age_group,
        COUNT(*) as passengers,
        ROUND(AVG(Survived)*100,1) as survival_pct
    FROM titanic
    WHERE Age IS NOT NULL
    GROUP BY age_group
    ORDER BY passengers DESC
    LIMIT 5
"""
    )
)

# Q14
print("\n--- Q14: Passengers travelling alone vs with family ---")
print(
    sql(
        """
    SELECT
        CASE WHEN SibSp + Parch = 0 THEN 'Alone'
             WHEN SibSp + Parch <= 3 THEN 'Small family'
             ELSE 'Large family'
        END as group_type,
        COUNT(*) as passengers,
        ROUND(AVG(Survived)*100,1) as survival_pct
    FROM titanic
    GROUP BY group_type
    ORDER BY passengers DESC
"""
    )
)

# Q15
print("\n--- Q15: Survival rate by title (Mr, Mrs, Miss, Master) ---")
print(
    sql(
        """
    SELECT
        CASE 
            WHEN Name LIKE '%Mr.%' THEN 'Mr'
            WHEN Name LIKE '%Mrs.%' THEN 'Mrs'
            WHEN Name LIKE '%Miss.%' THEN 'Miss'
            WHEN Name LIKE '%Master.%' THEN 'Master'
            ELSE 'Other'
        END as title,
        COUNT(*) as passengers,
        ROUND(AVG(Survived)*100,1) as survival_pct
    FROM titanic
    GROUP BY title
    ORDER BY survival_pct DESC
"""
    )
)

       PRACTICE SQL QUESTIONS

--- Q12: How many passengers embarked from each port? ---
  Embarked  passengers  survival_pct
0        S         644          33.7
1        C         168          55.4
2        Q          77          39.0

--- Q13: Top 5 most common age groups ---
   age_group  passengers  survival_pct
0       20.0         220          35.0
1       30.0         167          43.7
2       10.0         102          40.2
3       40.0          89          38.2
4        0.0          62          61.3

--- Q14: Passengers travelling alone vs with family ---
     group_type  passengers  survival_pct
0         Alone         537          30.4
1  Small family         292          57.9
2  Large family          62          16.1

--- Q15: Survival rate by title (Mr, Mrs, Miss, Master) ---
    title  passengers  survival_pct
0     Mrs         125          79.2
1    Miss         182          69.8
2  Master          40          57.5
3   Other          27          44.4
4      Mr         51

## 7. Key Takeaways — Day 12 🎯

### SQL Fundamentals
- SQL = Structured Query Language — talks to databases
- DuckDB lets us run SQL directly on Pandas DataFrames!
- SQL reads almost like English — SELECT this FROM here WHERE condition

### SELECT & WHERE
- `SELECT col1, col2 FROM table` — choose columns
- `WHERE condition` — filter rows BEFORE grouping
- `ORDER BY col DESC` — sort results
- `LIMIT n` — return only n rows

### GROUP BY & HAVING
- `GROUP BY col` — group rows by unique values
- Aggregate functions: `COUNT()`, `SUM()`, `AVG()`, `MIN()`, `MAX()`
- `HAVING condition` — filter groups AFTER aggregation
- KEY RULE: WHERE filters rows, HAVING filters groups!

### JOINs
| Join Type | Result |
|---|---|
| INNER JOIN | Only matching rows from both tables |
| LEFT JOIN | All left rows + matching right rows |
| RIGHT JOIN | All right rows + matching left rows |
| FULL JOIN | All rows from both tables |

### Subqueries
- **In WHERE:** `WHERE col > (SELECT AVG(col) FROM table)`
- **In FROM:** Query the result of another query
- **With IN:** `WHERE col IN (SELECT col FROM table WHERE...)`

### CASE WHEN
- SQL's version of if/elif/else
- Used to create categories from raw data
- `CASE WHEN x > 50 THEN 'High' WHEN x > 20 THEN 'Medium' ELSE 'Low' END`

### Key Titanic SQL Insights
- Cherbourg passengers had best survival (55.4%) — more 1st class!
- Small families survived best (57.9%) vs alone (30.4%)
- Mrs title: 72% survival vs Mr: only 15.7%
- Babies (0-9): 61.3% survival — children first!

### Tools Used
- `duckdb.connect()` — create in-memory SQL engine
- `con.register('name', df)` — register DataFrame as SQL table
- `con.execute(query).df()` — run SQL and get DataFrame back